# Experiment: A State-Gated Recurrent Cell


**Hypothesis:** Standard recurrent models have powerful but specific update mechanisms. We can design a custom recurrent cell that allows for a highly flexible, state-dependent interaction between the previous state and the current input.

Our proposed recurrent update rule is composed of two parallel components:
$$
h_t = u(h_{t-1}, x_t) + v(h_{t-1}, x_t)
$$

1.  **Gated State Evolution `g(h, x)`:** This term is responsible for evolving the previous state `h_{t-1}`. It uses the current input `x_t` to create an attention-like gate that is applied directly to the previous state. This is analogous to the "forget gate" in an LSTM, allowing the model to selectively preserve or scale down information from its memory.
    $$
    u(h, x) = \text{softmax}\left(\frac{(h W_{hQ}) \cdot (x W_{hK})}{\sqrt{d_k}}\right) \cdot h
    $$

2.  **Gated Input Injection `h(h, x)`:** This term is responsible for introducing new information from the input `x_t`. It uses the previous state `h_{t-1}` to create a gate that is applied to a *projection* of the input. This allows the model to decide how much of the new information to write into the state, based on its current context.
    $$
    v(h, x) = \text{softmax}\left(\frac{(h W_{xQ}) \cdot (x W_{xK})}{\sqrt{d_k}}\right) \cdot (xW_{xV})
    $$

**Implications:**

- **Richer Dynamics:** This structure creates a rich, non-linear interaction where both the preservation of old information and the injection of new information are dynamically controlled by a combination of the current state and input.
- **Suitability for Short Sequences:** While this custom cell might be challenging to train on very long sequences, it is well-suited for tasks involving shorter sequences, such as processing a series of sentence embeddings, where its expressive power can be fully leveraged.

Let's formalize this by creating a `StateGatedRNNCell` module.

In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as nf
from torch.utils.data import DataLoader

import torchvision.datasets as datasets
import torchvision.transforms as transforms

from einops import rearrange, repeat, einsum
import math

# Define the parameters for our layers
batch_size = 1  # B
model_dim = 8  # D
num_heads = 4  # N
state_dim = model_dim // num_heads  # S
in_features = model_dim
out_features = model_dim

x = torch.randn(batch_size, model_dim)
h = torch.randn(batch_size, state_dim, num_heads)

# Query from state 'h', Key from input 'x', Value from input 'x'
W_xQ = nn.Linear(model_dim, model_dim, bias=False)  # Query Proj
W_xK = nn.Linear(model_dim, model_dim, bias=False)  # Key Proj
W_xV = nn.Linear(model_dim, model_dim, bias=False)  # Value Proj

# 1. Generate Query, Key, Value
x_Q = W_xQ(h.contiguous().view(batch_size, model_dim)).view(
    batch_size, state_dim, num_heads
)  # Shape: (B, D, N)
x_K = W_xK(x).view(batch_size, state_dim, num_heads)  # Shape: (B, D, N)
x_V = W_xV(x).view(batch_size, state_dim, num_heads)  # Shape: (B, D, N)

print(f"x shape:\n {x.shape}")
print(f"state shape:\n {h.shape}")
print(f"x_Q shape:\n {x_Q.shape}")
print(f"x_K shape:\n {x_K.shape}")
print(f"x_V shape:\n {x_V.shape}")

# 2. Calculate Attention Gate
# We need x_K to be (B, D, N) to interact with x_Q.
# We expand it, so each of the D channels in the query attends to the same key.
# x_K_expanded = x_K.unsqueeze(1).expand_as(x_Q) # Shape: (B, D, N)
# print(f"x_K:\n {x_K}")
# print(f"x_K_expanded shape:\n {x_K_expanded.shape}")
# print(f"x_K_expanded:\n {x_K_expanded}")

# Calculate gate using element-wise product and softmax, just like in your g(h,x)
attn_gate = nf.softmax(x_Q * x_K, dim=-1)  # Shape: (B, D, N)

# 3. Apply Gate to Value
# We need x_V to be (B, D, N). We expand it so the same value is available to each channel's gate.
# x_V_expanded = x_V.unsqueeze(1).expand_as(attn_gate) # Shape: (B, D, N)

x_out_refined_multi_head = attn_gate * x_V  # Shape: (B, D, N)

print(f"Output tensor state shape:\n {x_out_refined_multi_head.shape}")
print(f"Output tensor state:\n {x_out_refined_multi_head}")

x_out_refined = x_out_refined_multi_head.contiguous().view(
    batch_size, model_dim
)

x shape:
 torch.Size([1, 8])
state shape:
 torch.Size([1, 2, 4])
x_Q shape:
 torch.Size([1, 2, 4])
x_K shape:
 torch.Size([1, 2, 4])
x_V shape:
 torch.Size([1, 2, 4])
Output tensor state shape:
 torch.Size([1, 2, 4])
Output tensor state:
 tensor([[[-0.1630,  0.0380, -0.0627,  0.1157],
         [ 0.0311, -0.0685,  0.0399,  0.0445]]], grad_fn=<MulBackward0>)


In [44]:
# Projections: Query from state 'h', Key from input 'x', Value from state 'h'
W_hQ = nn.Linear(model_dim, model_dim, bias=False) # Query Proj
W_hk = nn.Linear(model_dim, model_dim, bias=False) # Key Proj

# 1. Generate Query, Key, Value
h_Q = W_hQ(h.contiguous().view(batch_size, model_dim)).view(batch_size, state_dim, num_heads) # Shape: (B, D, N)
h_K = W_hk(x).view(batch_size, state_dim, num_heads) # Shape: (B, D, N)

print(f"x shape:\n {x.shape}")
print(f"state shape:\n {h.shape}")
print(f"h_Q shape:\n {h_Q.shape}")
print(f"h_K shape:\n {h_K.shape}")

# 2. Calculate Attention Gate
# The Key 'h_K' from the input is expanded to match the dimensions of the Query 'h_Q' from the state.
# This allows the input to gate every channel of the state evolution.
attn_gate = nf.softmax(h_Q * h_K, dim=-1)

# 3. Apply Gate to Value
# The gate is applied element-wise to the Value projection of the state.
# This produces the 'g(h,x)' term of our recurrence.
sf_h = attn_gate * h

print(f"Output tensor state shape:\n {sf_h.shape}")
print(f"Output tensor state:\n {sf_h}")


x shape:
 torch.Size([1, 8])
state shape:
 torch.Size([1, 2, 4])
h_Q shape:
 torch.Size([1, 2, 4])
h_K shape:
 torch.Size([1, 2, 4])
Output tensor state shape:
 torch.Size([1, 2, 4])
Output tensor state:
 tensor([[[-0.1062,  0.0784,  0.3338, -0.2822],
         [-0.1665, -0.1520, -0.2169,  0.3533]]], grad_fn=<MulBackward0>)


In [30]:

class StateGatedRNNCell(nn.Module):
    def __init__(self, model_dim, num_heads):
        super().__init__()
        self.model_dim = model_dim # D
        self.num_heads = num_heads # N

        # --- g(h,x) projections: Gated state evolution ---
        self.W_hQ = nn.Linear(model_dim * num_heads, model_dim * num_heads, bias=False)
        self.W_hK = nn.Linear(model_dim, model_dim * num_heads, bias=False)

        # --- h(h,x) projections: Gated input injection ---
        self.W_xQ = nn.Linear(model_dim * num_heads, model_dim * num_heads, bias=False)
        self.W_xK = nn.Linear(model_dim, model_dim * num_heads, bias=False)
        self.W_xV = nn.Linear(model_dim, model_dim * num_heads, bias=False)

        
        # Final output projection to combine heads
        self.W_O = nn.Linear(model_dim * num_heads, model_dim, bias=False)
        
        # Layer Normalization for the hidden state
        self.norm = nn.LayerNorm(model_dim * num_heads)

    def forward(self, x, h):
        """
        x: current input token -> (batch, model_dim)
        h: previous state -> (batch, model_dim, num_heads)
        """
        batch_size = h.shape[0]

        # Normalize the hidden state before it's used for gating
        h = h.contiguous().view(batch_size, self.num_heads * self.model_dim)
        h = self.norm(h)

        # --- 1. Calculate g(h,x): The gated previous state ---
        g_Q = self.W_hQ(h).view(batch_size, self.model_dim, self.num_heads)
        g_K = self.W_hK(x).view(batch_size, self.model_dim, self.num_heads)
        g_attn_gate = nf.softmax(g_Q * g_K, dim=-1)

        h_reshaped = h.view(batch_size, self.model_dim, self.num_heads)
        g_out = g_attn_gate * h_reshaped

        # --- 2. Calculate h(h,x): The gated new input ---
        h_Q = self.W_xQ(h).view(batch_size, self.model_dim, self.num_heads)
        h_K = self.W_xK(x).view(batch_size, self.model_dim, self.num_heads)
        h_V = self.W_xV(x).view(batch_size, self.model_dim, self.num_heads)
        
        h_attn_gate = nf.softmax(h_Q * h_K, dim=-1)
        h_out = h_attn_gate * h_V
    
        # --- 3. Combine heads and produce final state ---
        h_next = g_out + h_out
        output = self.W_O(h_next.view(batch_size, self.model_dim * self.num_heads))

        print(f"g_out shape: {g_out.shape}")
        print(f"h_out shape: {h_out.shape}")
        
        return output, h_next

In [32]:

# --- Let's test the cell ---
batch_size = 1
model_dim = 8
num_heads = 4

# Dummy input
x_t = torch.randn(batch_size, model_dim)
h_prev = torch.randn(batch_size, model_dim, num_heads)

# Initialize and run the cell
cell = StateGatedRNNCell(model_dim, num_heads)
output, h_next = cell(x_t, h_prev)

print(f"Previous state shape: {h_prev.shape}")
print(f"Next state shape:     {h_next.shape}")

g_out shape: torch.Size([1, 8, 4])
h_out shape: torch.Size([1, 8, 4])
Previous state shape: torch.Size([1, 8, 4])
Next state shape:     torch.Size([1, 8, 4])


In [142]:
# class MultiHeadStateGatedRNNCell(nn.Module):
#     def __init__(self, input_dim, state_dim, num_heads):
#         super().__init__()
#         assert state_dim % num_heads == 0, "state_dim must be divisible by num_heads"
        
#         self.input_dim = input_dim
#         self.state_dim = state_dim
#         self.num_heads = num_heads
#         self.head_dim = state_dim // num_heads

#         # --- g(h,x) projections: Gated state evolution ---
#         # We use one large linear layer and split the output into heads for efficiency
#         self.W_hQ = nn.Linear(state_dim, state_dim, bias=False)
#         self.W_hK = nn.Linear(input_dim, state_dim, bias=False)

#         # --- h(h,x) projections: Gated input injection ---
#         self.W_xQ = nn.Linear(state_dim, state_dim, bias=False)
#         self.W_xK = nn.Linear(input_dim, state_dim, bias=False)
#         self.W_xV = nn.Linear(input_dim, state_dim, bias=False)

#         # Final output projection to combine heads
#         self.W_O = nn.Linear(state_dim, state_dim, bias=False)
        
#         # Layer Normalization for the hidden state
#         self.norm = nn.LayerNorm(state_dim)

#     def forward(self, x, h):
#         """
#         x: current input token -> (batch, input_dim)
#         h: previous state -> (batch, input_dim, state_dim)
#         """
#         batch_size, channels, _ = h.shape
        
#         # Normalize the hidden state before use
#         h_norm = self.norm(h)
        
#         # --- 1. Calculate g(h,x): The gated previous state ---
#         g_Q = self.W_hQ(h_norm).view(batch_size, channels, self.num_heads, self.head_dim)
#         g_K = self.W_hK(x).view(batch_size, 1, self.num_heads, self.head_dim)
        
#         # Expand key to match query dimensions for gating
#         g_K_expanded = g_K.expand_as(g_Q)
#         g_attn_gate = nf.softmax(g_Q * g_K_expanded, dim=-1)
        
#         # Reshape h for multi-head gating
#         h_reshaped = h.view(batch_size, channels, self.num_heads, self.head_dim)
#         g_out = g_attn_gate * h_reshaped

#         # --- 2. Calculate h(h,x): The gated new input ---
#         h_Q = self.W_xQ(h_norm).view(batch_size, channels, self.num_heads, self.head_dim)
#         h_K = self.W_xK(x).view(batch_size, 1, self.num_heads, self.head_dim)
#         h_V = self.W_xV(x).view(batch_size, 1, self.num_heads, self.head_dim)

#         h_K_expanded = h_K.expand_as(h_Q)
#         h_V_expanded = h_V.expand_as(h_Q)
        
#         h_attn_gate = nf.softmax(h_Q * h_K_expanded, dim=-1)
#         h_out = h_attn_gate * h_V_expanded
        
#         # --- 3. Combine heads and produce final state ---
#         combined_heads = (g_out + h_out).view(batch_size, channels, self.state_dim)
#         h_next = self.W_O(combined_heads)

#         return h_next

In [53]:
# --- 4. Build the Encoder-Decoder Model ---


class Encoder(nn.Module):
    def __init__(self, input_dim, num_heads):
        super().__init__()
        self.input_dim = input_dim
        # self.cell = StateGatedRNNCell(input_dim, state_dim)
        self.cell = StateGatedRNNCell(input_dim, num_heads)

    def forward(self, x_seq):
        """
        x_seq: (batch, seq_len, input_dim)
        Returns the final hidden state of the encoder.
        """
        batch_size, seq_len, _ = x_seq.shape
        # The state is now (batch, input_dim, state_dim), which is unusual.
        # Let's keep it for now but note it's a place for future refinement.
        h = torch.zeros(batch_size, self.input_dim, num_heads, device=x_seq.device)
        for t in range(seq_len):
            h = self.cell(x_seq[:, t, :], h)
        return h


# Second, create the Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, input_dim, embedding_dim, num_heads):
        super().__init__()
        self.output_dim = output_dim
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim

        # An embedding layer for the decoder's input (which are class indices)
        self.embedding = nn.Embedding(self.output_dim, self.embedding_dim)

        # self.cell = StateGatedRNNCell(embedding_dim, self.state_dim)
        self.cell = StateGatedRNNCell(embedding_dim, num_heads)

        # It flattens the state and projects it to a single vector.
        self.readout_proj = nn.Linear(self.input_dim * self.num_heads, self.num_heads)

        # The final layer maps the state to logits over the possible output indices
        self.fc = nn.Linear(self.num_heads, self.output_dim)

    def forward(self, x_t, h_prev):
        """
        x_t: The previous token (either from target or last prediction) -> (batch, output_dim)
        h_prev: The previous hidden state from the decoder -> (batch, input_dim, state_dim)
        """
        # Embed the input token index
        x_embedded = self.embedding(x_t.squeeze(1))

        h_next = self.cell(x_embedded, h_prev)
        # Project state to logits
        batch_size = h_next.shape[0]
        flattened_state = h_next.view(batch_size, -1)
        readout_vector = self.readout_proj(flattened_state)

        # Project the learned readout vector to the final output logits
        output_logits = self.fc(readout_vector)

        return output_logits, h_next


In [50]:

# --- 5. Design a Proof-of-Concept Task: Sequence Sorting ---

# combine encoder and decoder into a single model
# The model will receive a sequence of numbers and must output the sorted sequence.
# This requires memory and reasoning about the order of elements.
class EncoderDecoderSorter(nn.Module):
    def __init__(self, input_dim, output_dim, embedding_dim, num_heads):
        super().__init__()
        self.encoder = Encoder(input_dim, num_heads)
        self.decoder = Decoder(output_dim, input_dim, embedding_dim, num_heads)
        
    def forward(self, x_input):
        batch_size, seq_len, _ = x_input.shape
        
        # --- Encoder Pass ---
        # The encoder produces a single context vector (the final hidden state)
        encoder_hidden = self.encoder(x_input)
        
        # --- Decoder Pass ---
        decoder_hidden = encoder_hidden
        outputs = []
        
        # Start token is the index 0
        decoder_input = torch.zeros(batch_size, 1, dtype=torch.long, device=x_input.device)
        
        for t in range(seq_len):
            decoder_output_logits, decoder_hidden = self.decoder(decoder_input, decoder_hidden)
            outputs.append(decoder_output_logits)
            
            # Get the predicted class index
            top1 = decoder_output_logits.argmax(1).unsqueeze(1)
            
            decoder_input = top1

        return torch.stack(outputs, dim=1).squeeze(-1)

In [51]:
# --- 6. Training and Evaluation for Index Prediction ---


def generate_data(batch_size, seq_len):
    unsorted = torch.rand(batch_size, seq_len)
    # The target is now the *indices* of the sorted values
    sorted_indices = torch.argsort(unsorted, dim=1)
    return unsorted, sorted_indices


# Hyperparameters
n_epochs = 200000
learning_rate = 0.001
batch_size = 64
sequence_length = 7
embedding_dim = 32
num_heads = 8

weight_decay_val = 1e-4
clip_value = 1.0

# Model, Loss, Optimizer
sorter_model = EncoderDecoderSorter(
    input_dim=1,
    output_dim=sequence_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(sorter_model.parameters(), lr=learning_rate, weight_decay=weight_decay_val)
# This will reduce the learning rate by a factor of 0.5 if the loss does not improve for 2000 epochs.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, "min", patience=2000, factor=0.5
)


In [52]:
print("Starting training on the Encoder-Decoder Sorter...")
running_loss = 0.0
min_loss = float("inf")
for epoch in range(n_epochs):
    # Generate new data for each epoch
    inputs, targets = generate_data(batch_size, sequence_length)

    # The model expects tensors of shape (batch, seq, features)
    inputs = inputs.unsqueeze(-1)

    # Forward pass
    outputs = sorter_model(inputs)
    loss = criterion(outputs.view(-1, sequence_length), targets.view(-1))

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(sorter_model.parameters(), clip_value)
    optimizer.step()

    if loss.item() < min_loss:
        min_loss = loss.item()

    running_loss += loss.item()
    if (epoch + 1) % 1000 == 0:
        avg_loss = running_loss / 1000

        print(
            f"Epoch [{epoch + 1}/{n_epochs}],"
            f" Loss: {loss.item():.4f},"
            f" Avg Loss: {avg_loss:.4f},"
            f" Min Loss: {min_loss:.4f}"
            f" LR: {optimizer.param_groups[0]['lr']:.6f},"
        )

        running_loss = 0.0

print("\nTraining finished.")

# --- 7. Evaluation ---
print("Evaluating the trained model...")
test_input, test_target = generate_data(1, sequence_length)
sorter_model.eval()
with torch.no_grad():
    test_input, test_target = generate_data(1, sequence_length)
    # For evaluation, never use teacher forcing
    test_output_logits = sorter_model(test_input.unsqueeze(-1))
    test_predictions = test_output_logits.argmax(2)
    probabilities = nf.softmax(test_output_logits, dim=2)
    confidence_scores = torch.gather(probabilities, 2, test_predictions.unsqueeze(-1)).squeeze()


print(f"\nInput Sequence Values: {test_input.numpy().flatten()}")
print(f"Target (Sorted Indices): {test_target.numpy().flatten()}")
print(f"Model Prediction (Indices): {test_predictions.numpy().flatten()}")
print(f"Model Confidence: {[f'{c:.2f}' for c in confidence_scores.numpy()]}")

# To make it more intuitive, let's see the sorted values based on the model's predicted indices
predicted_sorted_values = test_input[0][test_predictions[0]].numpy()
print(f"Values sorted by model prediction: {predicted_sorted_values}")


Starting training on the Encoder-Decoder Sorter...


RuntimeError: shape '[64, 8]' is invalid for input of size 8192